# Inference using a Pix2Pix model
* This notebook expects the images images to be only of the mask domain
* It will then load these images, run them through the generator and store them under a `generated/` folder located under the parent folder of `opt.dataroot`

### Defining the options

In [9]:
from types import SimpleNamespace
from data.single_dataset import SingleDataset

opt = SimpleNamespace()
opt.checkpoint_path = "checkpoints/wgan.pth"
opt.dataroot = "/mnt/c/Users/samaniegotorres.ma/datasets/unet/wgan/images"
opt.max_dataset_size = float("inf")
opt.direction = "AtoB"  # Change depending on how you trained your model
opt.input_nc = 3
opt.output_nc = 3
opt.preprocess = "resize_and_crop"
opt.load_size = 512
opt.crop_size = 512
opt.no_flip = True
# Don't modify these (unless you trained pix2pix without using default arguments)
opt.norm = "batch"
opt.ngf = 64
opt.netG = "unet_256"
opt.use_dropout = True
opt.init_gain = 0.02
opt.gpu_ids = [0]
dataset = SingleDataset(opt)

### Loading the model

In [10]:
import torch
from models.networks import define_G
from collections import OrderedDict

model_dict = torch.load(opt.checkpoint_path)
new_dict = OrderedDict()
for k, v in model_dict.items():
    # load_state_dict expects keys with prefix 'module.'
    new_dict["module." + k] = v

# make sure you pass the correct parameters to the define_G method
generator = define_G(input_nc=opt.input_nc, output_nc=opt.output_nc, ngf=opt.ngf, netG=opt.netG,
                     norm=opt.norm, use_dropout=opt.use_dropout, init_gain=opt.init_gain,
                     gpu_ids=opt.gpu_ids)
generator.load_state_dict(new_dict)


initialize network with normal


<All keys matched successfully>

In [11]:
from util.util import tensor2im, save_image
from pathlib import Path
from tqdm import tqdm

output_dir = Path(opt.dataroot).parent / "generated"
output_dir.mkdir(exist_ok=True)
print(f"Processing {len(dataset)} images with the trained Pix2Pix generator ...")
for img in tqdm(dataset):
    tensor = img["A"].unsqueeze(0)
    generated_img = tensor2im(generator(tensor))
    img_name = img["A_paths"].split("/")[-1]
    save_image(generated_img, output_dir / img_name)
print(f"Generated images stored under: {output_dir}")

Processing 400 images with the trained Pix2Pix generator ...


100%|██████████| 400/400 [00:59<00:00,  6.75it/s]

Generated images stored under: /mnt/c/Users/samaniegotorres.ma/datasets/unet/wgan/generated
